# Apache SpamAssassin Spam Classification - Data Cleaning and Exploratory Data Analysis

A folderwise train test split was made. The headers from the traning set were inspected. There are spam related header flags which have to be modifed to prevent ground truth leakge. Further header inspection was done but it was complex due to inconsistent frequency of headers in an email.

In [1]:
from pathlib import Path
import pandas as pd

from utils import get_data_info, detect_duplicates, create_train_test_split

In [2]:
seed = 42
RAW_DATA_DIR = Path('./raw_data')

## 1. Remove Duplicates

removing duplicate emails identified in `01_initial_eda.ipynb`. If not removed, this can cause data leakage from train test split.

In [3]:
detect_duplicates(RAW_DATA_DIR, remove_duplicates=True)

data_info, msc_info = get_data_info(RAW_DATA_DIR)
data_info

Total number of duplicates: 1580
Total files: 9165
Total spam files: 2388
Total ham files: 6777
Percentage of spam: 26.06%

After removing duplicates:
Total files (w/o duplicates): 9165
Total spam files (w/o duplicates): 2388
Total ham files (w/o duplicates): 6777
Percentage of spam: 26.06%


,folder,type,files_count_w_duplicates,files_count_w/o_duplicates
0,20030228_easy_ham_2,ham,1400,1400
1,20030228_hard_ham,ham,250,250
2,20030228_spam,spam,500,500
4,20030228_easy_ham,ham,2500,2500
5,20030228_spam_2,spam,1397,1397
6,20021010_spam,spam,491,491
7,20021010_hard_ham,ham,217,217
8,20021010_easy_ham,ham,2410,2410
9,20050311_spam_2,spam,0,0


## 2. Train-Test Split

We will use a 80-20 split. For each folder in data, we will do a 80-20 split of the folder. The split will be saved on `data_split` folder.

In [4]:
DATA_SPLIT_DIR = Path('./data_split')

X_train, y_train, X_test, y_test = create_train_test_split(RAW_DATA_DIR, DATA_SPLIT_DIR)
len(X_train), len(X_test), len(y_train), len(y_test)

(7330, 1835, 7330, 1835)

In [5]:
y_train.count(1) + y_train.count(0), y_test.count(1) + y_test.count(0)

(7330, 1835)

In [6]:
assert(len(X_train) == len(y_train))
assert(len(X_test) == len(y_test))

## 3. Header Inspection

Now that the train test split has been done. We will only use the training set to do any inspections.

We will check spam related headers if any? and what information they contain.

However, our judgement might be influenced by test set, so we will do these inspection in `02_data_cleaning_&_eda.ipynb`.

In [7]:
# TODO: how many of those spam headers are there?
# TODO: are there any headers with SPAM in them? (e.g. X-Spam-Flag, X-Spam-Status, X-Spam-Level, X-Spam-Score, X-Spam-Checker-Version, etc.)

from email import policy
from email.parser import BytesParser
from collections import defaultdict

train_folder = Path("./data_split/train/")
headers = set()
spam_headers = set()

for folder in train_folder.iterdir():
    for idx, filepath in enumerate(folder.iterdir()): 
        with open(filepath, "rb") as f:
            msg = BytesParser(policy=policy.default).parse(f)

        for header in msg.keys():
            if header in headers:
                continue
            headers.add(header)

            if 'spam' in header.lower():
                spam_headers.add(header)
                # msg.replace_header(header, "HIDDEN")
                # print(msg)
                print(folder)
                print(header)
                print(msg[header])
                print()

data_split/train/20030228_easy_ham_2
X-Spamadvice
Pursuant to US Code; Title 47; Chapter 5; Subchapter II; 227 any unsolicited commercial email to this address will be subject to a download and archival fee of US$500.  Pursuant to California Business & Professions Code; S17538.45 any email service provider involved in SPAM activities will be liable for statutory damages of US$50 per message, up to US$25,000 per day.  Pursuant to California Penal Code; S502 any unsolicited email sent to this address is in violation of Federal Law and is subject to fine and/or imprisonment.



data_split/train/20030228_easy_ham_2
X-spam
0 

data_split/train/20030228_hard_ham
X-Spam
spam  MAILTO_LINK, FREQ_SPAM_PHRASE, CTYPE_JUST_HTML, RCVD_IN_OSIRUSOFT_COM

data_split/train/20030228_hard_ham
X-MailScanner-SpamCheck
not spam, SpamAssassin (score=-5.1, required 5,	PGP_SIGNATURE)

data_split/train/20030228_easy_ham
X-Evi-Mailscanner-Spamcheck
not spam (whitelisted), SpamAssassin    (score=-11, required 5.5, IN_REP_TO, SOURCEFORGE_MENTIONED,    SPAMASSASSIN_MENTIONED, SPAM_PHRASE_00_01, USER_IN_WHITELIST_TO)

data_split/train/20030228_easy_ham
X-Mailscanner-Spamcheck
not spam, SpamAssassin ()

data_split/train/20030228_easy_ham
X-Spam-Level


data_split/train/20030228_spam_2
X-SpamCop-Checked
192.168.1.15 211.213.123.8 

data_split/train/20030228_spam_2
X-SpamCop-Disposition
Blocked bl.spamcop.net

data_split/train/20021010_hard_ham
X-Spam-Status
No, hits=-32.7 required=5.0	tests=ALL_CAP_PORN,AWL,EMAIL_MARKETING,HABEAS_SWE,	      LINES_OF_YELLING,NO_FEE,OPT_IN,T_NONSENSE_FROM_40

In [8]:
headers = list(headers)
for header in headers:
    print(header)

X-Originatingip
X_Id
Reply-BY
X-Stormpost-To
Return-Path
X-Originating-IP
X-Uri
X-Habeas-Swe-3
Comments
X-MimeOLE
X-Pmflags
X-Mimeole
X-Envelope-To
X-SpamCop-Disposition
X-Dumb-Filters
X-Delivery-Agent
X-Wigglefluff
X-Author
X-Platter
X-list
X-No-Archive
X-Mailscanner-Spamcheck
X-AWeber-Remote-Host
X-Uidl
Dcc
X-PMG-Msgid
X-Unsubscription-Info
X-Remove-Address
X-Gnupg-Publickey
X-Excuse
Errors-to
X-IMAPbase
X-Egroups-From
X-Airmail-Delivered
In-Reply-To
X-Gnupg-Keyid
X-Echelon-Distraction
X-Habeas-Swe-9
X-MDRCPT-To
X-Stock-Tip
Thread-Index
NNTP-Posting-Host
X-MS-Tnef-Correlator
X-SLUIDL
X-Mas
X-Spam-Status
Delivered-To
X-Mailscanner
X-Delivered-To
X-Note
X-Loop-Detect
X-Bofh
X-Keywords
X-GPG-Keyserver
X-Copyright
X-Face
X-PGP-Key_2
X-Razor-Id
X-Habeas-Swe-5
X-Complaints-To
>Received
X-Subliminal-Message
MIME-Version
X-Acceptable-Languages
X-Content_id
X-Copyright(C)
X-Expiredinmiddle
X-From-Line
X-Info
X-Priority
X-Spam-Checker-Version
Resent-To
X-Gnupg-Fingerprint
X-MailScanner
Reply-t

are any headers linked with a class?

### Conditional Prevalence of Header by Class

We will observe how often a header H is prevalent in the dataset with respect to the classes of 'Ham' and 'Spam'. In other words, we will compute $P(H|Spam)$ and $P(H|Ham)$.

$P(H|Spam) = n_{H_{spam}}/n_{spam}$ and $P(H|ham) = n_{H_{ham}}/n_{ham}$


In [9]:
from email import policy
from email.parser import BytesParser
from collections import defaultdict

train_folder = Path("./data_split/train/")
# TODO: implement this
headers_count = {'spam': defaultdict(int), 'ham': defaultdict(int)}

for folder in train_folder.iterdir():
    for idx, filepath in enumerate(folder.iterdir()): 
        with open(filepath, "rb") as f:
            msg = BytesParser(policy=policy.default).parse(f)

        seen = set()
        for header in msg.keys():
            if header in seen:
                continue
            seen.add(header)
            if 'spam' in folder.name.lower():
                headers_count['spam'][header] += 1
            else:
                headers_count['ham'][header] += 1

In [10]:
# TODO: which header is linked to which class? (spam or ham) - can be used to identify spammy headers and/or words in the header.

num_spam = sum(y_train)
num_ham = len(y_train) - num_spam

header_df = pd.DataFrame.from_dict(headers_count)
header_df['Prob_H_given_Spam'] = header_df['spam'] / num_spam
header_df['Prob_H_given_Ham'] = header_df['ham'] / num_ham
header_df

,spam,ham,Prob_H_given_Spam,Prob_H_given_Ham
Return-Path,1736.0,5413.0,0.909377,0.998524
Delivered-To,1298.0,5022.0,0.679937,0.926397
Received,1908.0,5316.0,0.999476,0.980631
Message-Id,1655.0,5113.0,0.866946,0.943184
To,1899.0,5277.0,0.994762,0.973437
...,...,...,...,...
X-Spam-Filter,NaN,1.0,NaN,0.000184
X-Authenticated-Sender,NaN,1.0,NaN,0.000184
Forwarded,NaN,5.0,NaN,0.000922
X-Newsgroups,NaN,2.0,NaN,0.000369


Since, there can be multiple headers that have the same name in the same email, thus inflating the conditional probability. Further, some headers are rare and do not appear in most emails but could be good indicators of ham or spam while some other generic headers like From, To, etc apppear in an email mandatorily, making it hard for us to inspect which of these headers are actually correlated with each other.

## 5. Summary

In [11]:
summary = pd.DataFrame({
    'Findings': [
                'Duplicate Emails Removed',
                'Ground Truth Leakage Concern',
                'Header inspection was done.'
                ],
    'Details': [
                'Removed.',
                'SPAM flags will be set to "HIDDEN" in the pipeline to avoid ground truth leakage.',
                'Headers were inspected using conditional prevalence but it was too complex and abandoned as of now.'
                ]
})

summary

,Findings,Details
0,Duplicate Emails Removed,Removed.
1,Ground Truth Leakage Concern,"SPAM flags will be set to ""HIDDEN"" in the pipe..."
2,Header inspection was done.,Headers were inspected using conditional preva...
